In [1]:
# Import required libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import tensorflow as tf
from tensorflow.keras import layers, models
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
# Load and examine data
data = pd.read_csv('datas/final_heart_data.csv')
print(f"Dataset shape: {data.shape}")
print("\nFirst few rows of the data:")
data.head()

Dataset shape: (8763, 24)

First few rows of the data:


,Unnamed: 0,Age,Sex,Cholesterol,Heart Rate,Diabetes,Family History,Smoking,Obesity,Alcohol Consumption,...,Stress Level,Sedentary Hours Per Day,Income,BMI,Triglycerides,Physical Activity Days Per Week,Sleep Hours Per Day,systolic,diastolic,Heart Attack Risk
0,0,67,1,208,72,0,0,1,0,0,...,9,6.615001,261404,31.251233,286,0,6,158,88,0
1,1,21,1,389,98,1,1,1,1,1,...,1,4.963459,285768,27.194973,235,1,7,165,93,0
2,2,21,0,324,72,1,0,0,0,0,...,9,9.463426,235282,28.176571,587,4,4,174,99,0
3,3,84,1,383,73,1,1,1,0,1,...,9,7.648981,125640,36.464704,378,3,4,163,100,0
4,4,66,1,318,93,1,1,1,1,0,...,6,1.514821,160555,21.809144,231,1,5,91,88,0


In [3]:
# Data Preparation and Analysis
# Prepare features and target
X = data.drop(['Heart Attack Risk'], axis=1)
y = data['Heart Attack Risk']

# Check class distribution
print("Class distribution:")
print(y.value_counts())
print("\nClass distribution (percentage):")
print(y.value_counts(normalize=True) * 100)

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("\nData preparation completed!")
print(f"Training set shape: {X_train_scaled.shape}")
print(f"Test set shape: {X_test_scaled.shape}")

Class distribution:
Heart Attack Risk
0    5624
1    3139
Name: count, dtype: int64

Class distribution (percentage):
Heart Attack Risk
0    64.178934
1    35.821066
Name: proportion, dtype: float64

Data preparation completed!
Training set shape: (7010, 23)
Test set shape: (1753, 23)


In [4]:
# Calculate class weights properly
class_counts = y_train.value_counts()
total_samples = len(y_train)
class_weights = {
    0: total_samples / (2 * class_counts[0]),
    1: total_samples / (2 * class_counts[1])
}
# Increase weight for class 1
class_weights[1] *= 1.5

print("Class weights:", class_weights)

# Build the neural network
model = models.Sequential([
    layers.Dense(32, activation='relu', input_shape=(X_train.shape[1],)),
    layers.Dropout(0.2),
    layers.Dense(16, activation='relu'),
    layers.Dense(1, activation='sigmoid')
])

# Compile the model
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# Display model summary
model.summary()

Class weights: {0: 0.7790620137808402, 1: 2.0937873357228196}


C:\Users\ravik\AppData\Roaming\Python\Python312\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 32)             │           768 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,313 (5.13 KB)

 Trainable params: 1,313 (5.13 KB)

 Non-trainable params: 0 (0.00 B)

In [6]:
# Train the model with class weights
history = model.fit(
    X_train_scaled, 
    y_train,
    epochs=20,
    batch_size=32,
    validation_split=0.2,
    class_weight=class_weights,
    verbose=1
)

KeyError: 0

In [ ]:
# Evaluate model
test_loss, test_accuracy = model.evaluate(X_test_scaled, y_test)
print(f'\nTest accuracy: {test_accuracy:.4f}')

# Make predictions
y_pred = model.predict(X_test_scaled)
y_pred_classes = (y_pred > 0.5).astype(int)

# Print classification report
from sklearn.metrics import classification_report
print('\nClassification Report:')
print(classification_report(y_test, y_pred_classes))

In [ ]:
# Visualize results
plt.figure(figsize=(12, 4))

# Plot training history - Loss
plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Model Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

# Plot training history - Accuracy
plt.subplot(1, 2, 2)
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Model Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

plt.tight_layout()
plt.show()